# FraudDetectAI — Credit Card Fraud Detection

Comparing machine learning approaches on a real-world, highly imbalanced dataset 
of European credit card transactions (September 2013, ULB). All features V1–V28 
are PCA-transformed for confidentiality. Data is loaded from PostgreSQL.

## Dataset at a glance
| | |
|---|---|
| Total transactions | 284,807 |
| Fraudulent cases | 492 (0.172%) |
| Features | 30 (V1–V28, Time, Amount, Class) |
| Transaction window | 2 days |

## Approaches compared
- **SMOTE** — synthetic oversampling of the minority class before classification
- **Class weight balancing** — penalises minority misclassification during training
- **Isolation Forest** — anomaly detection, no label balancing required

## Evaluation strategy
Accuracy is misleading on imbalanced data — a model predicting *no fraud* every 
time scores 99.8%. We use AUPRC, ROC-AUC, F1, and SHAP for explainability.

---
*Data source: [ULB Machine Learning Group — Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)*

In [2]:
import os
import warnings

# Data
import numpy as np
import pandas as pd

# DB 
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

# Imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Models
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Hyperparameter tuning
import optuna
from optuna.samplers import TPESampler

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    RocCurveDisplay,
)

# Explainability
import shap

# Settings
SEED = 42
TEST_SIZE = 0.2

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

## Database Connection

In [7]:
load_dotenv()

DB_URL = (
    f"postgresql://"
    f"{os.getenv('POSTGRES_USER', 'postgres')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'postgres')}@"
    f"localhost:5432/"
    f"{os.getenv('POSTGRES_DB', 'frauddetect')}"
)

engine = create_engine(DB_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM transactions;"))
    count = result.scalar()
    
print(f"Total transactions in the database: {count}")

Total transactions in the database: 284807


**Get Column Names**:

In [11]:
with engine.connect() as conn:
    res = conn.execute(text("SELECT * FROM transactions WHERE 1=0;"))

print("Columns in the dataset:")
print(res.keys())

Columns in the dataset:
RMKeyView(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class'])


**Class Distribution**

In [22]:
# ── Class distribution ────────────────────────────────────────────────────────
query = """
    SELECT 
        "Class",
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 4) AS percentage
    FROM transactions
    GROUP BY "Class"
    ORDER BY "Class";
"""

with engine.connect() as conn:
    class_dist = pd.read_sql(text(query), conn)

print("Class distribution:\n")
print(class_dist.to_string(index=False))

Class distribution:

 Class  count  percentage
     0 284315     99.8273
     1    492      0.1727
